In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [47]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from urllib.request import urlopen
import math
import time
import requests
import pandas as pd

kospi, kosdak, divident = 'gsum', 'ksd_gsum', 'div'

In [102]:
call_nav_fin_gsum(kospi)
call_nav_fin_gsum(kosdak)
call_nav_fin_divident(divident)

kospi_item_list = load_csv_file(kospi)
kosdak_item_list = load_csv_file(kosdak)
divident_item_list = load_csv_file(divident)

kosdak_item_list.tail(3)

selected_divident_items = divident_item_list[(divident_item_list['수익률(%)']>=5.0) &
                                             (divident_item_list['ROE(%)']>=5.0) &
                                             (divident_item_list['PER(배)']>=10.0) &
                                             (divident_item_list['PBR(배)']>=1.0) &
                                             (divident_item_list['배당성향(%)']>=25.0)]
display(selected_divident_items)
p_kospi = kospi_item_list[kospi_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
p_kosdak = kosdak_item_list[kosdak_item_list['전일비변동']=='하락'].sort_values('등락률', ascending=False)
proposed_kospi = pd.merge(selected_divident_items, p_kospi, how='inner', on=['종목명'])
proposed_kosdak = pd.merge(selected_divident_items, p_kosdak, how='inner', on=['종목명'])

display(proposed_kospi.T)
display(proposed_kosdak.T)


~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 4시트 데이터를 수집 하였습니다~~
~~ 총 10페이지 데이터를 수집 하였습니다~~


,종목명,현재가,기준월,배당금,수익률(%),배당성향(%),ROE(%),PER(배),PBR(배),과거3년배당금_Max,1년전,2년전,3년전
6,크레버스,14250,25.04,1500,10.53,167.06,17.99,21.71,3.06,2000,2000,1800,2000
8,이크레더블,15840,24.12,1590,10.04,148.28,27.34,12.07,3.18,780,780,1040,2720
16,정상제이엘에스,6150,24.12,530,8.62,94.60,9.05,12.55,1.08,530,530,530,530
39,케이카,15750,24.12,1150,7.30,126.03,19.09,14.04,2.73,760,760,760,750
51,현대엘리베이터,79500,24.12,5500,6.92,108.40,14.42,11.18,1.51,800,4000,500,800
65,디지털대성,7450,24.12,500,6.71,86.41,10.45,12.59,1.36,300,200,200,300
79,한국쉘석유,420500,24.12,27000,6.42,95.76,28.93,11.30,3.25,25000,25000,18000,19000
97,진양폴리,4080,24.12,250,6.13,79.02,9.74,20.99,2.03,250,250,200,175
111,효성ITX,12500,24.12,750,6.00,75.93,16.71,12.90,2.06,750,750,750,750
114,진로발효,18590,24.12,1100,5.92,71.02,12.59,10.92,1.29,650,650,450,1150


,0,1
종목명,케이카,현대엘리베이터
현재가_x,15750,79500
기준월,24.12,24.12
배당금,1150,5500
수익률(%),7.3,6.92
배당성향(%),126.03,108.4
ROE(%),19.09,14.42
PER(배),14.04,11.18
PBR(배),2.73,1.51
과거3년배당금_Max,760,800


,0
종목명,디지털대성
현재가_x,7450
기준월,24.12
배당금,500
수익률(%),6.71
배당성향(%),86.41
ROE(%),10.45
PER(배),12.59
PBR(배),1.36
과거3년배당금_Max,300


In [101]:
def load_csv_file(str):
    date = time.strftime("%y%m%d", time.localtime())
    item_list = pd.read_csv(f'data/naver/fin_{str}_{date}.csv', encoding='cp949') 
    return item_list

## 네이버 금융/주식/시가총액 화면

In [100]:
def call_nav_fin_gsum(string):

    driver = webdriver.Chrome()
    if(string == 'gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=0')
    elif(string == 'ksd_gsum'):
        driver.get('https://finance.naver.com/sise/sise_market_sum.naver?sosok=1')
    time.sleep(0.5) # 초기화면 접근

    no_tgl_bxs = len(driver.find_elements(By.CSS_SELECTOR, 'td > input[type="checkbox"]'))
    toggle_boxes = [f'option{i}' for i in range(1, no_tgl_bxs+1)] # 체크박스 이름 준비

    pageN = 10
    max_tgl = 6
    item_list = pd.DataFrame([])
    for page in range(1,pageN+1):
        page_item_list = []
        for sheet in range(0, math.ceil(no_tgl_bxs/max_tgl)):
            dflt_toggle_boxes = driver.find_elements(By.CSS_SELECTOR, 'td.choice > input[type="checkbox"]')
            [elem.send_keys(Keys.SPACE) for elem in dflt_toggle_boxes]
            time.sleep(0.5) # 디폴트 체크박스 toggle

            sheet_toggle_boxes = toggle_boxes[sheet*max_tgl:min((sheet+1)*max_tgl, no_tgl_bxs+1)]
            new_toggle_boxes = [select_box for select_box in sheet_toggle_boxes]
            [driver.find_element(By.ID, box_id).send_keys(Keys.SPACE) for box_id in new_toggle_boxes]
            time.sleep(0.5) # 시트별 새로운 체크박스 toggle
            driver.find_element(By.CSS_SELECTOR, 'div.item_btn > a').click()
            time.sleep(0.5) # 토글 체크후 submit버튼 클릭

            soup = BeautifulSoup(driver.page_source, 'html.parser')
            table_head = [e.text.strip() for e in soup.select('thead > tr >th')][:-1]
            table_head.insert(3, '전일비변동')
            items = soup.select('tbody > tr') # 테이블 헤드 수집
            sheet_item_list = []
            for idx, item in enumerate(items):
                if(item.select_one('td.no')):
                    no = item.select_one('td.no').text
                    title = item.select_one('a.tltle').text
                    price_vals = [e.text.strip() for e in item.select('td.number')]
                    if(price_vals[1].strip()=='보합0'):
                        price_vals[1]=['보합',0]
                    else:
                        price_vals[1] = [price_vals[1].split('\n')[0], price_vals[1].split('\n')[1].strip()]

                    price_vals.insert(0, no)
                    price_vals.insert(1, title)
                    temp = price_vals[3][1]
                    price_vals.insert(3,price_vals[3][0])
                    price_vals[4] = temp

                    sheet_item_list.append(price_vals) # 테이블 내용 수집 
            sheet_item_df = pd.DataFrame(sheet_item_list, columns=table_head)

            if sheet == 0:
                page_item_list = sheet_item_df
            elif sheet < round(no_tgl_bxs/max_tgl):
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:13]]], axis=1)
            else: 
                page_item_list = pd.concat([page_item_list, sheet_item_df[table_head[7:9]]], axis=1)
                # 페이지 내 시트 축적
        item_list = pd.concat([item_list, page_item_list], axis=0) # 페이지 내용 축적

        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5) # next page로
    print(f'~~ 총 {page}페이지 {sheet}시트 데이터를 수집 하였습니다~~')

    item_list.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')
    item_list.sample(10)

## 네이버 배당 화면 접근

In [99]:
def call_nav_fin_divident(string):
    driver = webdriver.Chrome()
    driver.get('https://finance.naver.com/sise/dividend_list.naver')
    time.sleep(0.5)

    pageN = 10
    item_list = []
    for page in range(1,pageN+1):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        table_head = [e.text.strip() for e in soup.select('thead > tr > th')]
        table_head[9] = '과거3년배당금_Max'
        


        items = soup.select('table.type_1.tb_ty> tbody > tr')
        for idx, item in enumerate(items):
            if(item.text.strip()):
                price_vals = [e.text.strip().replace(',','').replace('-','0') for e in item.select('td')]
                price_vals.insert(9, max(price_vals[9:12]))

                item_list.append({table_head[i]:e for i, e in enumerate(price_vals)})

        # next page로
        page_div = driver.find_element(By.CSS_SELECTOR, 'table.Nnavi > tbody > tr')
        if(page<pageN): # 지정 마지막 페이지 초과(pageN+1) 이동 금지
            page_div.find_element(By.LINK_TEXT, str(page+1)).click()
            time.sleep(1.5)
            
    print(f'~~ 총 {page}페이지 데이터를 수집 하였습니다~~')
    df = pd.DataFrame(item_list)
    df.to_csv(f'data/naver/fin_{string}_{time.strftime("%y%m%d", time.localtime())}.csv', index=False, encoding='cp949')